In [1]:
import numpy as np
import pandas as pd
from neutrality_test_bootstrap import *

In [7]:
raw = pd.read_csv("Table-S7-sgRNA-Raw-Counts.csv")
sample_names = raw.iloc[0]
sgRNAs = raw.iloc[1:, 0].astype(str).to_numpy()

In [8]:
def counts_for_cols(cols):
    """Return a (K x len(cols)) DataFrame of integer counts with index=normalized sgRNAs."""
    mat = raw.loc[1:, cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    mat.index = sgRNAs
    return mat.astype(np.int64)

j0 = raw.columns.get_loc("preinfection samples for 24 hpi of murine pneumonia model")
pre_cols = list(raw.columns[j0 : j0 + 2])  # (Pre1, Pre2)

**Test for Dox-group**

In [ ]:
# ---- donor = Pre2 (from 'preinfection samples for 24 hpi...') ----
donor_col = next(c for c in pre_cols if str(sample_names[c]) == "Pre2")
donor_counts = counts_for_cols([donor_col]).iloc[:, 0]  # pd.Series, index=sgRNA####

# ---- recipients = 24 hpi mice WITH dox (Mice_plusDox_*) ----
i0 = raw.columns.get_loc("24 hpi of murine pneumonia model")
i1 = raw.columns.get_loc("preinfection samples for 48 hpi of murine pneumonia model")
cols_24hpi = list(raw.columns[i0:i1])

plusdox_cols = [c for c in cols_24hpi if isinstance(sample_names[c], str) and sample_names[c].startswith("Mice_plusDox_")]

recipient_counts_df = counts_for_cols(plusdox_cols).T
recipient_counts_df.index = [sample_names[c] for c in plusdox_cols]   # mice IDs # recipient_counts_df.columns are already normalized sgRNA####

In [ ]:
results, diagnostics = neutrality_lrt_bootstrap(donor_counts, recipient_counts_df,
   # bottleneck=Nb_vec,  # One value per host, in recipient row order
    reestimate_bottleneck=True, n_boot=300, n_jobs=2, seed=12345,)
results.to_csv("neutrality_bootstrap_Dox.csv", index=False)

Fitting observed bottlenecks and computing observed statistics...
Bootstrap 1/300; elapsed 32.9 s
Bootstrap 10/300; elapsed 124.4 s
Bootstrap 20/300; elapsed 240.5 s
Bootstrap 30/300; elapsed 357.1 s
Bootstrap 40/300; elapsed 472.7 s
Bootstrap 50/300; elapsed 588.5 s
Bootstrap 60/300; elapsed 703.7 s
Bootstrap 70/300; elapsed 818.2 s
Bootstrap 80/300; elapsed 933.7 s
Bootstrap 90/300; elapsed 1050.9 s
Bootstrap 100/300; elapsed 1166.6 s
Bootstrap 110/300; elapsed 1281.5 s
Bootstrap 120/300; elapsed 1399.0 s
Bootstrap 130/300; elapsed 1515.6 s
Bootstrap 140/300; elapsed 1631.1 s
Bootstrap 150/300; elapsed 1748.7 s
Bootstrap 160/300; elapsed 1863.8 s
Bootstrap 170/300; elapsed 1979.8 s
Bootstrap 180/300; elapsed 2093.2 s
Bootstrap 190/300; elapsed 2208.2 s
Bootstrap 200/300; elapsed 2323.7 s
Bootstrap 210/300; elapsed 2439.6 s
Bootstrap 220/300; elapsed 2555.4 s
Bootstrap 230/300; elapsed 2674.7 s
Bootstrap 240/300; elapsed 2787.9 s
Bootstrap 250/300; elapsed 2905.0 s
Bootstrap 260/300; 

**Test for no-Dox-group**

In [ ]:
# ---- donor = Pre1 (from 'preinfection samples for 24 hpi...') ----
donor_col = next(c for c in pre_cols if str(sample_names[c]) == "Pre1")
donor_counts = counts_for_cols([donor_col]).iloc[:, 0]  # pd.Series, index=sgRNA####

# ---- recipients = 24 hpi mice WITH dox (Mice_noDox_*) ----
i0 = raw.columns.get_loc("24 hpi of murine pneumonia model")
i1 = raw.columns.get_loc("preinfection samples for 48 hpi of murine pneumonia model")
cols_24hpi = list(raw.columns[i0:i1])

nodox_cols = [c for c in cols_24hpi if isinstance(sample_names[c], str) and sample_names[c].startswith("Mice_noDox_")]

recipient_counts_df = counts_for_cols(nodox_cols).T
recipient_counts_df.index = [sample_names[c] for c in nodox_cols]   # mice IDs # recipient_counts_df.columns are already normalized sgRNA####

In [ ]:
results, diagnostics = neutrality_lrt_bootstrap(donor_counts, recipient_counts_df,
  #  bottleneck=Nb_vec,  # One value per host, in recipient row order
    reestimate_bottleneck=True, n_boot=300, n_jobs=2, seed=12345,)
results.to_csv("neutrality_bootstrap_noDox.csv", index=False)